# Mesh Tutorial 3: Discrete Calculus and Differential Geometry

This tutorial covers the mathematical operations available in PhysicsNeMo-Mesh:

1. **Gradients**: Compute spatial derivatives of scalar and vector fields
2. **Divergence**: Measure the "outflow" of vector fields
3. **Curl**: Measure the "rotation" of vector fields (3D only)
4. **Curvature**: Gaussian and mean curvature at vertices
5. **Intrinsic vs Extrinsic**: Derivatives in tangent space vs ambient space
6. **Vector Calculus Identities**: Verify curl(grad) = 0 and div(curl) = 0

---

## Why Discrete Calculus Matters for Physics-AI

In physics-informed machine learning, we often need to:

- **Compute PDE residuals**: Requires gradients, divergence, Laplacian on mesh data
- **Extract geometric features**: Curvature, normals, gradients as model inputs
- **Enforce physics constraints**: Conservation laws involve divergence
- **Loss functions on fields**: Compare predicted vs. actual field gradients

PhysicsNeMo-Mesh provides GPU-accelerated, differentiable implementations of these
operators, enabling gradient-based optimization through mesh-based physics.

In [ ]:
import torch
import math

from physicsnemo.mesh import Mesh
from physicsnemo.mesh.primitives.surfaces import sphere_icosahedral, torus
from physicsnemo.mesh.primitives.planar import unit_square
from physicsnemo.mesh.primitives.volumes import cube_volume

## Section 1: Computing Gradients

The gradient of a scalar field tells you the direction of steepest increase.

PhysicsNeMo-Mesh supports two methods:

| Method | Description | Best For |
|--------|-------------|----------|
| `lsq` | Weighted least-squares reconstruction | General use, robust on irregular meshes |
| `dec` | Discrete Exterior Calculus | Mathematically rigorous, geometric problems |

For most applications, `lsq` (the default) works well.

In [ ]:
# Create a 2D mesh
mesh = unit_square.load(subdivisions=5)

# Create a scalar field: T = x + 2y
# The exact gradient should be [1, 2]
mesh.point_data["T"] = mesh.points[:, 0] + 2 * mesh.points[:, 1]

# Compute gradient using least-squares
mesh_with_grad = mesh.compute_point_derivatives(keys="T", method="lsq")

# Access the computed gradient
grad_T = mesh_with_grad.point_data["T_gradient"]
print(f"Gradient shape: {grad_T.shape}")
print(f"Sample gradient values (should be ~[1, 2]):")
print(grad_T[:5])

In [ ]:
# Verify the gradient is accurate
expected = torch.tensor([1.0, 2.0])
mean_grad = grad_T.mean(dim=0)
error = (mean_grad - expected).norm()
print(f"Expected gradient: {expected}")
print(f"Mean computed gradient: {mean_grad}")
print(f"Error: {error:.6f}")

### Gradients of Vector Fields (Jacobian)

For vector fields, the gradient is a matrix (the Jacobian).

In [ ]:
# Create a vector field: v = [x*y, x^2]
# Jacobian: [[y, x], [2x, 0]]
mesh = unit_square.load(subdivisions=5)
x, y = mesh.points[:, 0], mesh.points[:, 1]
mesh.point_data["v"] = torch.stack([x * y, x**2], dim=-1)

# Compute Jacobian
mesh_with_jac = mesh.compute_point_derivatives(keys="v", method="lsq")
jacobian = mesh_with_jac.point_data["v_gradient"]

print(f"Jacobian shape: {jacobian.shape}  (n_points, n_output_dims, n_spatial_dims)")
print(f"\nJacobian at point (x=0.5, y=0.5):")
# Find point near (0.5, 0.5)
idx = ((mesh.points - torch.tensor([0.5, 0.5])).norm(dim=-1)).argmin()
print(f"  Location: {mesh.points[idx]}")
print(f"  Jacobian:\n{jacobian[idx]}")
print(f"  Expected: [[0.5, 0.5], [1.0, 0.0]]")

### Computing Multiple Gradients at Once

You can compute gradients of multiple fields in a single call.

In [ ]:
mesh = unit_square.load(subdivisions=5)
mesh.point_data["pressure"] = mesh.points[:, 0]**2 + mesh.points[:, 1]**2
mesh.point_data["temperature"] = torch.sin(math.pi * mesh.points[:, 0])

# Compute gradients of both fields
mesh_grad = mesh.compute_point_derivatives(keys=["pressure", "temperature"])

print("Computed gradient fields:")
for key in mesh_grad.point_data.keys():
    if "gradient" in key:
        print(f"  {key}: {mesh_grad.point_data[key].shape}")

## Section 2: Divergence

The divergence of a vector field measures the net "outflow" at each point.

For a 2D field v = [v_x, v_y]: div(v) = ∂v_x/∂x + ∂v_y/∂y

In [ ]:
from physicsnemo.mesh.calculus import compute_divergence_points_lsq

# Create a vector field with known divergence
# v = [x, y] has divergence = 2 (constant)
mesh = unit_square.load(subdivisions=5)
velocity = mesh.points.clone()  # v = [x, y]

div_v = compute_divergence_points_lsq(mesh, velocity)

print(f"Divergence shape: {div_v.shape}")
print(f"Mean divergence: {div_v.mean():.4f} (expected: 2.0)")
print(f"Std divergence: {div_v.std():.6f}")

In [ ]:
# A solenoidal (divergence-free) field
# v = [-y, x] is a rotation field with div = 0
mesh = unit_square.load(subdivisions=5)
x, y = mesh.points[:, 0], mesh.points[:, 1]
rotation_field = torch.stack([-y, x], dim=-1)

div_rotation = compute_divergence_points_lsq(mesh, rotation_field)
print(f"Divergence of rotation field: {div_rotation.mean():.6f} (expected: 0)")

## Section 3: Curl (3D Only)

The curl measures the "rotation" or "vorticity" of a vector field.

curl(v) = [∂v_z/∂y - ∂v_y/∂z, ∂v_x/∂z - ∂v_z/∂x, ∂v_y/∂x - ∂v_x/∂y]

Curl is only defined in 3D.

In [ ]:
from physicsnemo.mesh.calculus import compute_curl_points_lsq

# Create a 3D mesh
mesh = cube_volume.load(n=8)
# Use the boundary surface for better visualization
mesh = mesh.get_boundary_mesh().subdivide(1, "linear")

# A rotation field around the z-axis: v = [-y, x, 0]
# Its curl is [0, 0, 2] (constant)
x, y, z = mesh.points[:, 0], mesh.points[:, 1], mesh.points[:, 2]
rotation_field = torch.stack([-y, x, torch.zeros_like(z)], dim=-1)

curl_v = compute_curl_points_lsq(mesh, rotation_field)

print(f"Curl shape: {curl_v.shape}")
print(f"Mean curl: {curl_v.mean(dim=0)} (expected: [0, 0, 2])")

## Section 4: Curvature

PhysicsNeMo-Mesh computes two types of curvature for surface meshes:

| Curvature | Formula | Properties |
|-----------|---------|------------|
| **Gaussian** (K) | K = κ₁ × κ₂ | Intrinsic, preserved under bending |
| **Mean** (H) | H = (κ₁ + κ₂) / 2 | Extrinsic, depends on embedding |

where κ₁ and κ₂ are the principal curvatures.

In [ ]:
# For a sphere of radius r:
# - Gaussian curvature K = 1/r²
# - Mean curvature H = 1/r

radius = 2.0
sphere = sphere_icosahedral.load(radius=radius, subdivisions=4)

K = sphere.gaussian_curvature_vertices
H = sphere.mean_curvature_vertices

print(f"Sphere radius: {radius}")
print(f"\nGaussian curvature:")
print(f"  Expected: {1/radius**2:.4f}")
print(f"  Mean computed: {K.mean():.4f}")
print(f"\nMean curvature:")
print(f"  Expected: {1/radius:.4f}")
print(f"  Mean computed: {H.mean():.4f}")

In [ ]:
# Visualize curvature on the bunny
bunny = torch.load("assets/bunny.pt", weights_only=False).subdivide(2, "loop")

# Gaussian curvature: positive=convex (sphere-like), negative=saddle
bunny.point_data["K"] = bunny.gaussian_curvature_vertices
bunny.draw(point_scalars="K", cmap="RdBu", show_edges=False)

In [ ]:
# Mean curvature: useful for detecting ridges and valleys
bunny.point_data["H"] = bunny.mean_curvature_vertices
bunny.draw(point_scalars="H", cmap="coolwarm", show_edges=False)

### Gauss-Bonnet Theorem

The Gauss-Bonnet theorem relates total Gaussian curvature to topology:

∫ K dA = 2π × χ(M)

where χ is the Euler characteristic. For a closed surface: χ = 2 - 2g (g = genus/handles).

- Sphere (g=0): χ = 2, total K = 4π
- Torus (g=1): χ = 0, total K = 0

In [ ]:
from physicsnemo.mesh.geometry.dual_meshes import compute_dual_volumes_0

# Sphere: genus=0, χ=2, total K = 4π
sphere = sphere_icosahedral.load(subdivisions=4)
K = sphere.gaussian_curvature_vertices
dual_areas = compute_dual_volumes_0(sphere)
total_K = (K * dual_areas).sum()

print(f"Sphere (genus=0):")
print(f"  Expected total K: {4 * math.pi:.4f}")
print(f"  Computed total K: {total_K:.4f}")
print(f"  Error: {abs(total_K - 4*math.pi):.6f}")

In [ ]:
# Torus: genus=1, χ=0, total K = 0
donut = torus.load(major_radius=1.0, minor_radius=0.3, n_major=64, n_minor=32)
K_torus = donut.gaussian_curvature_vertices
dual_areas_torus = compute_dual_volumes_0(donut)
total_K_torus = (K_torus * dual_areas_torus).sum()

print(f"Torus (genus=1):")
print(f"  Expected total K: 0.0")
print(f"  Computed total K: {total_K_torus:.6f}")

## Section 5: Intrinsic vs Extrinsic Derivatives

For surfaces embedded in 3D, there are two types of derivatives:

| Type | Description | Use Case |
|------|-------------|----------|
| **Intrinsic** | Gradient in the tangent plane | Surface PDEs, physics on manifolds |
| **Extrinsic** | Gradient in ambient 3D space | Feature extraction, ambient flow |

Intrinsic gradients are perpendicular to the surface normal.

In [ ]:
# Create a sphere with a scalar field based on z-coordinate
sphere = sphere_icosahedral.load(subdivisions=3)
sphere.point_data["height"] = sphere.points[:, 2]

# Compute intrinsic gradient (in tangent space)
sphere_intrinsic = sphere.compute_point_derivatives(
    keys="height", method="lsq", gradient_type="intrinsic"
)
grad_intrinsic = sphere_intrinsic.point_data["height_gradient"]

# Compute extrinsic gradient (in ambient space)
sphere_extrinsic = sphere.compute_point_derivatives(
    keys="height", method="lsq", gradient_type="extrinsic"
)
grad_extrinsic = sphere_extrinsic.point_data["height_gradient"]

print(f"Intrinsic gradient shape: {grad_intrinsic.shape}")
print(f"Extrinsic gradient shape: {grad_extrinsic.shape}")

In [ ]:
# Verify: intrinsic gradient should be perpendicular to surface normal
normals = sphere.point_normals  # (n_points, 3)

# Dot product of gradient with normal should be ~0 for intrinsic
dot_intrinsic = (grad_intrinsic * normals).sum(dim=-1)
dot_extrinsic = (grad_extrinsic * normals).sum(dim=-1)

print(f"Intrinsic gradient · normal: {dot_intrinsic.abs().mean():.6f} (should be ~0)")
print(f"Extrinsic gradient · normal: {dot_extrinsic.abs().mean():.4f} (non-zero)")

## Section 6: Vector Calculus Identities

The discrete operators satisfy the fundamental vector calculus identities:

- **curl(grad(f)) = 0**: The curl of a gradient field is zero
- **div(curl(v)) = 0**: The divergence of a curl field is zero

Let's verify these numerically.

In [ ]:
from physicsnemo.mesh.calculus import compute_gradient_points_lsq
from physicsnemo.mesh.calculus import compute_curl_points_lsq
from physicsnemo.mesh.calculus import compute_divergence_points_lsq

# Create a 3D mesh (surface in 3D)
mesh = sphere_icosahedral.load(subdivisions=4)

# Scalar field: f = x² + y² + z²
f = (mesh.points ** 2).sum(dim=-1)

# Compute gradient
grad_f = compute_gradient_points_lsq(mesh, f)
print(f"grad(f) shape: {grad_f.shape}")

# Compute curl of gradient
curl_grad_f = compute_curl_points_lsq(mesh, grad_f)
print(f"curl(grad(f)) shape: {curl_grad_f.shape}")

# Should be approximately zero
print(f"\n|curl(grad(f))| mean: {curl_grad_f.norm(dim=-1).mean():.6f}")
print(f"|curl(grad(f))| max: {curl_grad_f.norm(dim=-1).max():.6f}")

In [ ]:
# div(curl(v)) = 0
# Create a vector field
v = mesh.points.clone()  # v = [x, y, z]

# Compute curl
curl_v = compute_curl_points_lsq(mesh, v)

# Compute divergence of curl
div_curl_v = compute_divergence_points_lsq(mesh, curl_v)

print(f"div(curl(v)) mean: {div_curl_v.mean():.6f}")
print(f"div(curl(v)) max: {div_curl_v.abs().max():.6f}")

## Section 7: Using Calculus for Physics-Informed Features

Here's a practical example: computing features for a physics-informed model.

In [ ]:
# Load a mesh representing some physical domain
mesh = torch.load("assets/bunny.pt", weights_only=False).subdivide(2, "loop")

# Simulate some physical fields
mesh.point_data["pressure"] = torch.sin(2 * math.pi * mesh.points[:, 0])
mesh.point_data["velocity"] = torch.randn(mesh.n_points, 3)

# Compute geometric features
mesh.point_data["gaussian_curvature"] = mesh.gaussian_curvature_vertices
mesh.point_data["mean_curvature"] = mesh.mean_curvature_vertices
mesh.point_data["normal"] = mesh.point_normals

# Compute field derivatives
mesh = mesh.compute_point_derivatives(keys=["pressure", "velocity"], method="lsq")

# Compute divergence of velocity
mesh.point_data["div_velocity"] = compute_divergence_points_lsq(
    mesh, mesh.point_data["velocity"]
)

print("Available features for ML model:")
for key in mesh.point_data.keys():
    shape = mesh.point_data[key].shape
    print(f"  {key}: {shape}")

## Summary

In this tutorial, you learned about discrete calculus on meshes:

1. **Gradients**: `compute_point_derivatives()` for scalar and vector fields
2. **Divergence**: `compute_divergence_points_lsq()` for vector fields
3. **Curl**: `compute_curl_points_lsq()` for 3D vector fields
4. **Curvature**: `gaussian_curvature_vertices`, `mean_curvature_vertices`
5. **Intrinsic vs Extrinsic**: `gradient_type="intrinsic"` for surface PDEs
6. **Identities**: curl(grad(f)) = 0, div(curl(v)) = 0

---

### Next Steps

- **Tutorial 4: Neighbors & Spatial Queries** - Adjacency, BVH, sampling
- **Tutorial 5: Quality & Repair** - Mesh validation and repair
- **Tutorial 6: ML Integration** - Performance, datapipes, torch.compile